# MLIP Active-Learning Run Generator

Generate ready-to-run LAMMPS folders from `POSCAR_*` and/or `*.vasp` files.

Outputs per structure:
- `atoms.data`
- `POSCAR`
- `al_md.in` (LAMMPS input with NVT/NPT and optional pressure baked in)


In [6]:
from __future__ import annotations

import random
import re
from pathlib import Path
from typing import Iterable

from ase.data import atomic_masses, atomic_numbers
from ase.io import read, write

DEFAULT_MODEL_YAML = '/nfshome/sicolo/work/MLIP/train/iter_01/output_potential.yaml'
DEFAULT_MODEL_ASI = '/nfshome/sicolo/work/MLIP/train/iter_01/output_potential.asi'


In [7]:
def unique_paths(patterns: Iterable[str], base_dir: Path) -> list[Path]:
    seen: set[Path] = set()
    found: list[Path] = []
    for pattern in patterns:
        for path in sorted(base_dir.glob(pattern)):
            if not path.is_file():
                continue
            rp = path.resolve()
            if rp not in seen:
                seen.add(rp)
                found.append(path)
    return found


def sanitize_name(path: Path) -> str:
    name = path.stem if path.suffix else path.name
    name = re.sub(r'^POSCAR_', '', name)
    # Shorten common exported-selection names:
    #   sel_019_surfaces_..._OUTCAR_0 -> surfaces_...
    name = re.sub(r'^sel_\d+_', '', name)
    name = re.sub(r'_OUTCAR_\d+$', '', name)
    safe = re.sub(r'[^A-Za-z0-9._-]+', '_', name).strip('._-')
    return safe or 'structure'


def first_seen_species(atoms) -> list[str]:
    ordered: list[str] = []
    for sym in atoms.get_chemical_symbols():
        if sym not in ordered:
            ordered.append(sym)
    return ordered


def choose_species_order(atoms, requested: list[str] | None) -> list[str]:
    present = set(atoms.get_chemical_symbols())
    if requested:
        missing = present.difference(requested)
        if missing:
            raise ValueError(f'Species order missing elements in structure: {sorted(missing)}')
        return [s for s in requested if s in present]
    return first_seen_species(atoms)


def mass_lines(species_order: list[str]) -> str:
    lines = []
    for i, sym in enumerate(species_order, start=1):
        z = atomic_numbers[sym]
        lines.append(f'mass {i} {atomic_masses[z]:.6f}  # {sym}')
    return '\n'.join(lines)


def fix_block(ensemble: str, t1: float, t2: float, tdamp: float, pressure: float | None, pdamp: float) -> str:
    if ensemble == 'nvt':
        return f'fix 1 all nvt temp {t1:g} {t2:g} {tdamp:g}'
    p = 0.0 if pressure is None else pressure
    return f'fix 1 all npt temp {t1:g} {t2:g} {tdamp:g} aniso {p:g} {p:g} {pdamp:g}'


def build_lammps_input(*, species_order, masses_text, ensemble, pressure, temp, end_temp, final_temp, seed, model_yaml, model_asi) -> str:
    f1 = fix_block(ensemble, temp, temp, 0.1, pressure, 1.0)
    f2 = fix_block(ensemble, temp, end_temp, 0.1, pressure, 1.0)
    f3 = fix_block(ensemble, end_temp, end_temp, 0.1, pressure, 1.0)
    f4 = fix_block(ensemble, end_temp, final_temp, 0.1, pressure, 1.0)
    species_text = ' '.join(species_order)
    mode_note = f'{ensemble.upper()} protocol'
    pressure_note = 'off' if ensemble == 'nvt' else f'{(0.0 if pressure is None else pressure):g}'

    return f'''units metal
boundary p p p
atom_style atomic

read_data atoms.data

{masses_text}

pair_style pace/extrapolation
pair_coeff * * {model_yaml} {model_asi} {species_text}

# Active-learning MD settings ({mode_note}; pressure={pressure_note})
variable randomSeed equal {seed}
variable timeStepLength equal 0.001
variable tDamp equal 0.1
variable pDamp equal 1.0

timestep ${{timeStepLength}}
velocity all create {temp:g} ${{randomSeed}}

fix pace_gamma all pair 1 pace/extrapolation gamma 1
compute max_pace_gamma all reduce max f_pace_gamma
variable dump_skip equal "c_max_pace_gamma < 5"
dump pace_dump all custom 5 extrapolative_structures.dump id type x y z f_pace_gamma
dump_modify pace_dump skip v_dump_skip
variable max_pace_gamma equal c_max_pace_gamma
fix extreme_extrapolation all halt 10 v_max_pace_gamma > 25

fix data1 all print 100 "$(step) $(temp) $(etotal)" append thermo_data.txt screen no
dump traj all atom 500 traj.xyz

{f1}
run 10000
unfix 1

{f2}
run 200000
unfix 1

{f3}
run 100000
unfix 1

{f4}
run 200000
unfix 1

print "$(pe)" file energy.dat
'''


## Edit Options

Set your structure folder and protocol here, then run the next cell.

In [8]:
CFG = {
    'src_dir': Path('./selected_structures'),              # folder containing POSCAR_* and/or *.vasp
    'patterns': ['POSCAR_*', '*.vasp'],
    'out_dir': Path('al_runs'),
    'ensemble': 'nvt',               # 'nvt' or 'npt'
    'pressure': None,                # e.g. 5000 or -5000; ignored for NVT
    'temp': 300.0,
    'end_temp': 1000.0,
    'final_temp': 300.0,
    'seed': None,                    # fixed int or None for random per structure
    'species_order': None,           # e.g. ['Li', 'Ni', 'O'] or None
    'model_yaml': DEFAULT_MODEL_YAML,
    'model_asi': DEFAULT_MODEL_ASI,
    'prefix': '',                    # optional folder prefix
}

CFG

{'src_dir': PosixPath('selected_structures'),
 'patterns': ['POSCAR_*', '*.vasp'],
 'out_dir': PosixPath('al_runs'),
 'ensemble': 'nvt',
 'pressure': None,
 'temp': 300.0,
 'end_temp': 1000.0,
 'final_temp': 300.0,
 'seed': None,
 'species_order': None,
 'model_yaml': '/nfshome/sicolo/work/MLIP/train/iter_01/output_potential.yaml',
 'model_asi': '/nfshome/sicolo/work/MLIP/train/iter_01/output_potential.asi',
 'prefix': ''}

In [9]:
def generate_runs(cfg: dict) -> list[Path]:
    src_dir = Path(cfg['src_dir']).resolve()
    out_dir = Path(cfg['out_dir'])
    if not out_dir.is_absolute():
        out_dir = (src_dir / out_dir).resolve()

    inputs = unique_paths(cfg['patterns'], src_dir)
    if not inputs:
        raise FileNotFoundError(f'No structure files found in {src_dir} for patterns {cfg["patterns"]}')

    ensemble = str(cfg['ensemble']).lower()
    if ensemble not in {'nvt', 'npt'}:
        raise ValueError('ensemble must be nvt or npt')

    pressure = cfg.get('pressure') if ensemble == 'npt' else None
    out_dir.mkdir(parents=True, exist_ok=True)

    created = []
    print(f'Found {len(inputs)} structures in {src_dir}')
    print(f'Writing runs to {out_dir}')
    print(f'Ensemble: {ensemble.upper()}')
    print(f'Pressure: {pressure if ensemble == "npt" else "off"}')

    for idx, src in enumerate(inputs, start=1):
        atoms = read(src)
        species_order = choose_species_order(atoms, cfg.get('species_order'))
        seed = cfg['seed'] if cfg.get('seed') is not None else random.randint(10, 9_999_999)

        run_name = f"{cfg.get('prefix', '')}s{idx:03d}_{sanitize_name(src)}"
        run_dir = out_dir / run_name
        run_dir.mkdir(parents=True, exist_ok=True)

        write(run_dir / 'atoms.data', atoms, format='lammps-data', specorder=species_order)
        write(run_dir / 'POSCAR', atoms, format='vasp')

        lmp_text = build_lammps_input(
            species_order=species_order,
            masses_text=mass_lines(species_order),
            ensemble=ensemble,
            pressure=pressure,
            temp=float(cfg['temp']),
            end_temp=float(cfg['end_temp']),
            final_temp=float(cfg['final_temp']),
            seed=int(seed),
            model_yaml=str(cfg['model_yaml']),
            model_asi=str(cfg['model_asi']),
        )
        (run_dir / 'al_md.in').write_text(lmp_text)

        created.append(run_dir)
        species_text = ' '.join(species_order)
        print(f'[{idx}/{len(inputs)}] {src.name} -> {run_dir.name} (species: {species_text}, seed: {seed})')

    print('\nRun LAMMPS inside any generated folder with: lmp -in al_md.in')
    return created


In [10]:
runs = generate_runs(CFG)
runs

Found 20 structures in /nfshome/sicolo/work/MLIP/train/al_workflow/selected_structures
Writing runs to /nfshome/sicolo/work/MLIP/train/al_workflow/selected_structures/al_runs
Ensemble: NVT
Pressure: off
[1/20] POSCAR_sel_001_surfaces_003_0ML_s1040_xy_k000_OUTCAR_0.vasp -> s001_surfaces_003_0ML_s1040_xy_k000 (species: Li Ni O, seed: 5816758)
[2/20] POSCAR_sel_002_defective_structures_antiphase_s1050_k001_OUTCAR_0.vasp -> s002_defective_structures_antiphase_s1050_k001 (species: Li Ni O, seed: 9574133)
[3/20] POSCAR_sel_003_binary_phases_LiO8_s1030_k001_OUTCAR_0.vasp -> s003_binary_phases_LiO8_s1030_k001 (species: Li O, seed: 5319403)
[4/20] POSCAR_sel_004_surfaces_NiO2_10-5_symm_s0980_xy_k000_OUTCAR_0.vasp -> s004_surfaces_NiO2_10-5_symm_s0980_xy_k000 (species: Ni O, seed: 4452193)
[5/20] POSCAR_sel_005_surfaces_003_067ML_s1050_xy_k000_OUTCAR_0.vasp -> s005_surfaces_003_067ML_s1050_xy_k000 (species: Li Ni O, seed: 5770939)
[6/20] POSCAR_sel_006_defective_structures_stacking_faults_P3_s10

[PosixPath('/nfshome/sicolo/work/MLIP/train/al_workflow/selected_structures/al_runs/s001_surfaces_003_0ML_s1040_xy_k000'),
 PosixPath('/nfshome/sicolo/work/MLIP/train/al_workflow/selected_structures/al_runs/s002_defective_structures_antiphase_s1050_k001'),
 PosixPath('/nfshome/sicolo/work/MLIP/train/al_workflow/selected_structures/al_runs/s003_binary_phases_LiO8_s1030_k001'),
 PosixPath('/nfshome/sicolo/work/MLIP/train/al_workflow/selected_structures/al_runs/s004_surfaces_NiO2_10-5_symm_s0980_xy_k000'),
 PosixPath('/nfshome/sicolo/work/MLIP/train/al_workflow/selected_structures/al_runs/s005_surfaces_003_067ML_s1050_xy_k000'),
 PosixPath('/nfshome/sicolo/work/MLIP/train/al_workflow/selected_structures/al_runs/s006_defective_structures_stacking_faults_P3_s1020_k001'),
 PosixPath('/nfshome/sicolo/work/MLIP/train/al_workflow/selected_structures/al_runs/s007_surfaces_003_083ML_s0950_xy_k001'),
 PosixPath('/nfshome/sicolo/work/MLIP/train/al_workflow/selected_structures/al_runs/s008_defective